In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('/app')
from database.db_connection import get_connection

conn = get_connection()
df = pd.read_sql("SELECT * FROM ml_features", conn)
conn.close()

print(f"Trước làm sạch: {df.shape}")

# ── Bỏ floor_number (100% null) ───────────────────────────────
df = df.drop(columns=["floor_number", "scraped_at", "listing_id"])

# ── Lọc giá bất hợp lý ───────────────────────────────────────
# Phòng trọ TP.HCM thực tế: 500k → 20 triệu/tháng
df = df[df["price_vnd"].between(500_000, 20_000_000)]

# ── Lọc diện tích bất hợp lý ─────────────────────────────────
# Phòng trọ thực tế: 6m² → 80m²
df = df[df["area_m2"].between(6, 80)]

# ── Lọc price_per_m2 bất hợp lý ──────────────────────────────
# Dưới 50k/m² hoặc trên 500k/m² là parse sai
df = df[df["price_per_m2"].between(50_000, 500_000)]

print(f"Sau làm sạch : {df.shape}")
print(f"Đã loại bỏ  : {992 - len(df)} dòng lỗi")



Trước làm sạch: (992, 15)
Sau làm sạch : (892, 12)
Đã loại bỏ  : 100 dòng lỗi


/tmp/ipykernel_424/3984546960.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM ml_features", conn)


In [2]:
# Fix lỗi encoding tiếng Việt từ SQL Server
# Nguyên nhân: SQL Server trả NVARCHAR nhưng pandas đọc sai encoding

# Cách 1: Map trực tiếp từ district_id (chắc chắn đúng)
DISTRICT_NAME_MAP = {
    1: "Quận 1",    2: "Quận 2",    3: "Quận 3",
    4: "Quận 4",    5: "Quận 5",    6: "Quận 6",
    7: "Quận 7",    8: "Quận 8",    9: "Quận 9",
    10: "Quận 10",  11: "Quận 11",  12: "Quận 12",
    13: "Bình Thạnh", 14: "Bình Tân",  15: "Gò Vấp",
    16: "Phú Nhuận", 17: "Tân Bình",  18: "Tân Phú",
    19: "Thủ Đức",  20: "Bình Chánh", 21: "Hóc Môn",
    22: "Nhà Bè",   23: "Cần Giờ",   24: "Củ Chi",
}

df["district_name"] = df["district_id"].map(DISTRICT_NAME_MAP)
print("Các quận có trong data:")
print(df["district_name"].value_counts())

Các quận có trong data:
district_name
Bình Thạnh    129
Tân Bình      111
Gò Vấp        111
Tân Phú        67
Thủ Đức        62
Quận 7         52
Quận 12        47
Quận 9         47
Quận 10        42
Bình Tân       37
Quận 3         32
Phú Nhuận      31
Quận 2         27
Quận 1         22
Quận 8         18
Quận 6         13
Quận 5         13
Quận 11        12
Nhà Bè          8
Bình Chánh      5
Hóc Môn         4
Quận 4          2
Name: count, dtype: int64


In [3]:
# ── Feature 1: Log transform giá ─────────────────────────────
# Giá bị skew lệch phải → log giúp model học tốt hơn
# log1p(x) = log(x+1) → an toàn khi x=0
df["log_price"]    = np.log1p(df["price_vnd"])
df["log_area"]     = np.log1p(df["area_m2"])

# ── Feature 2: Nhóm quận theo mức giá trung bình ─────────────
# Thay vì dùng district_id thô (1-24), nhóm thành 3 tier
district_avg = df.groupby("district_id")["price_vnd"].mean()

def classify_district(district_id):
    avg = district_avg.get(district_id, 0)
    if avg >= 4_500_000:
        return 2   # cao cấp: Q1, Q2, Q3, Q7
    elif avg >= 3_000_000:
        return 1   # trung bình: Q5, Bình Thạnh, Tân Bình
    else:
        return 0   # bình dân: Bình Chánh, Hóc Môn, Củ Chi

df["district_tier"] = df["district_id"].apply(classify_district)

tier_label = {0: "Bình dân", 1: "Trung bình", 2: "Cao cấp"}
print("Phân tier quận:")
print(df.groupby("district_tier")["price_vnd"].agg(["mean", "count"]).rename(
    index=tier_label
).rename(columns={"mean": "Giá TB", "count": "Số tin"}))

# ── Feature 3: Đếm số tiện ích ───────────────────────────────
amenity_cols = ["has_wc", "has_ac", "has_kitchen",
                "has_parking", "has_balcony", "has_security"]
df["amenity_count"] = df[amenity_cols].sum(axis=1)

print(f"\nSố tiện ích trung bình: {df['amenity_count'].mean():.1f}")
print(df["amenity_count"].value_counts().sort_index())

Phân tier quận:
                     Giá TB  Số tin
district_tier                      
Bình dân       2.853390e+06      59
Trung bình     3.576581e+06     651
Cao cấp        4.727198e+06     182

Số tiện ích trung bình: 0.9
amenity_count
0    375
1    294
2    154
3     57
4     10
5      2
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

# One-hot encode district_id (24 quận → 24 cột binary)
# Dùng pd.get_dummies thay vì LabelEncoder vì quận không có thứ tự
df_encoded = pd.get_dummies(df, columns=["district_id"], prefix="q", drop_first=True)

# Chọn feature cho model
FEATURE_COLS = [
    # Numeric
    "area_m2",
    "log_area",
    "price_per_m2",      # sẽ bỏ khi predict vì chứa thông tin giá
    "amenity_count",
    "district_tier",

    # Boolean tiện ích
    "has_wc", "has_ac", "has_kitchen",
    "has_parking", "has_balcony", "has_security",

    # One-hot quận
] + [c for c in df_encoded.columns if c.startswith("q_")]

TARGET_COL = "log_price"   # predict log(giá) → sau đó expm1() ra giá thật

# Bỏ price_per_m2 khỏi feature khi train thật
# (chứa thông tin giá → data leakage)
TRAIN_FEATURES = [f for f in FEATURE_COLS if f != "price_per_m2"]

X = df_encoded[TRAIN_FEATURES]
y = df_encoded[TARGET_COL]

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"\nFeature list ({len(TRAIN_FEATURES)}):")
for f in TRAIN_FEATURES:
    print(f"  {f}")

Shape X: (892, 31)
Shape y: (892,)

Feature list (31):
  area_m2
  log_area
  amenity_count
  district_tier
  has_wc
  has_ac
  has_kitchen
  has_parking
  has_balcony
  has_security
  q_2
  q_3
  q_4
  q_5
  q_6
  q_7
  q_8
  q_9
  q_10
  q_11
  q_12
  q_13
  q_14
  q_15
  q_16
  q_17
  q_18
  q_19
  q_20
  q_21
  q_22


In [5]:
# Lưu để dùng cho notebook train model
df_encoded.to_csv("/app/data/processed/features.csv",
                  index=False, encoding="utf-8-sig")

# Lưu riêng X và y
X.to_csv("/app/data/processed/X_train_raw.csv", index=False)
y.to_csv("/app/data/processed/y_train_raw.csv", index=False)

print(f"Đã lưu {len(df_encoded)} dòng")
print(f"Sẵn sàng cho Phase 4 — Train Model!")

# Tóm tắt nhanh
print(f"""
=== TÓM TẮT DATA SAU FEATURE ENGINEERING ===
Số dòng sạch    : {len(df_encoded)}
Số feature      : {len(TRAIN_FEATURES)}
Target          : log_price (sẽ expm1() ra VNĐ thật)
Giá trung bình  : {df['price_vnd'].mean()/1e6:.1f} triệu
Giá trung vị    : {df['price_vnd'].median()/1e6:.1f} triệu
Diện tích TB    : {df['area_m2'].mean():.0f} m²
Số tiện ích TB  : {df['amenity_count'].mean():.1f}
""")

Đã lưu 892 dòng
Sẵn sàng cho Phase 4 — Train Model!

=== TÓM TẮT DATA SAU FEATURE ENGINEERING ===
Số dòng sạch    : 892
Số feature      : 31
Target          : log_price (sẽ expm1() ra VNĐ thật)
Giá trung bình  : 3.8 triệu
Giá trung vị    : 3.6 triệu
Diện tích TB    : 25 m²
Số tiện ích TB  : 0.9

